# LamoLLM - Google Colab

A 1B parameter decoder-only transformer language model built from scratch with PyTorch.

**Steps:**
1. Clone the repo
2. Install dependencies
3. Train the model (pick a config size)
4. Generate text

**Runtime -> Change runtime type:**
- **GPU**: T4 (free) or A100
- **TPU**: TPU v5e-1 (free)

## 1. Setup & Clone Repo

In [2]:
!git clone https://github.com/arthurlamonattopro/LamoLLM.git /content/LamoLLM
%cd /content/LamoLLM

Cloning into '/content/LamoLLM'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 68 (delta 18), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 52.36 KiB | 7.48 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/LamoLLM


In [3]:
!pip install -r requirements.txt

### Save progress to Google Drive

Everything in `/content` is deleted when the runtime disconnects. Mounting Drive lets checkpoints survive crashes - training can resume from the last milestone instead of starting over.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_DIR = '/content/drive/MyDrive/LamoLLM/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)

Mounted at /content/drive
Checkpoints will be saved to: /content/drive/MyDrive/LamoLLM/checkpoints


In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")

DEVICE = "cpu"

if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    try:
        import torch_xla.core.xla_model as xm
        DEVICE = "tpu"
        print(f"TPU: {xm.get_xla_supported_devices(devkind='TPU')[0]}")
        print(f"TPU cores: {xm.xrt_world_size()}")
    except ImportError:
        print("No GPU or TPU detected. Using CPU.")

print(f"\nUsing device: {DEVICE}")

## 2. Training

Pick a config:
- **tiny** (~25M params) - quick test, runs in minutes
- **small** (~125M params) - moderate, ~30 min on T4
- **default** (~1.1B params) - full model, needs A100 or long run

Device is auto-detected (GPU > TPU > CPU).

**Quiet progress output:** one log line every **5%** of training (loss, ppl, lr, tok/s, ETA), and a checkpoint saved every **5%** (`lamollm_latest.pt`, rolling). The final model goes to `lamollm_final.pt`. Use `--keep_all_checkpoints` to retain every milestone file.

**Crash recovery:** with `--checkpoint_dir` pointed at Drive + `--auto_resume`, a crashed or disconnected session resumes from the last 5% milestone. Just re-run the training cell - at most one interval of progress is lost.

In [ ]:
# Checkpoint every 5% -> Drive | log line every 5% | auto-resume after crashes
!python scripts/train.py --config tiny --epochs 1 \
    --checkpoint_dir /content/drive/MyDrive/LamoLLM/checkpoints \
    --auto_resume

# If Colab crashes / disconnects mid-run: reconnect, re-run ALL cells above,
# then just re-run THIS cell. It reloads lamollm_latest.pt from Drive and
# continues where it stopped (LR schedule included).

# Options:
#   --checkpoint_every_pct 5     save cadence (% of run)
#   --log_every_pct 5            console log cadence (% of run)
#   --keep_all_checkpoints       keep lamollm_<pct>_step_<N>.pt files too
#   --no_eval                    skip validation-loss at each milestone

Initializing LamoLLM (tiny)...
Total parameters: 52,758,144
Trainable parameters: 52,758,144
Model size: 0.20 GB (FP32)
Loading dataset: Salesforce/wikitext (wikitext-103-raw-v1)...
README.md: 100% 10.5k/10.5k [00:00<00:00, 24.7MB/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:   0% 0.00/733k [00:00<?, ?B/s]
wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes: 100% 732k/732k [00:01<00:00, 513kB/s, 71.1kB/s  ]
wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file: 100% 733k/733k [00:01<00:00, 513kB/s, 71.2kB/s  ]

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:  38% 60.3M/157M [00:02<00:01, 62.6MB/s, 4.34MB/s  ]
wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:  33% 51.5M/157M [00:02<00:05, 20.8MB/s]
wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:  47% 73.3M/157M [00:02<00:01, 63.3MB/s, 6.71MB/s  ]
wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:  90% 142M/157M [00:03<00:00, 120MB/s,

## 3. Generate Text

In [ ]:
!python scripts/generate.py --checkpoint /content/drive/MyDrive/LamoLLM/checkpoints/lamollm_final.pt --prompt "Hello, I am" --max_tokens 100

## 4. Python API (Optional)

Use the generator directly in a cell.

In [ ]:
import sys
sys.path.insert(0, '/content/LamoLLM')

from inference.generator import LamoGenerator

generator = LamoGenerator.from_checkpoint('/content/drive/MyDrive/LamoLLM/checkpoints/lamollm_final.pt')

output = generator.generate('The future of AI is', max_new_tokens=100)
print(output)

## 5. Interactive Chat

In [ ]:
import sys
sys.path.insert(0, '/content/LamoLLM')

from inference.generator import LamoGenerator

generator = LamoGenerator.from_checkpoint('/content/drive/MyDrive/LamoLLM/checkpoints/lamollm_final.pt')

# Try a few prompts
prompts = [
    "Once upon a time",
    "The meaning of life is",
    "In a distant galaxy",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(f"Output: {generator.generate(prompt, max_new_tokens=80)}")
    print("-" * 50)